In [3]:
#1. PARAMETRIZATION

activities_with_predecessors = {
    'C': {'A'},
    'D': {'A', 'B'},
    'E': {'C'},
    'F': {'C'},
    'G': {'D', 'E'},
    'H': {'F', 'G'}
}

time_duration = {
    'A': 2,
    'B': 3,
    'C': 2,
    'D': 4,
    'E': 4,
    'F': 3,
    'G': 5,
    'H': 2
}

full_activities = set(activities_with_predecessors.keys()) | {'A', 'B'}

processing_order = ('A', 'B', 'C', 'D', 'E', 'F', 'G', 'H')

print(activities_with_predecessors)
print(time_duration)
print(full_activities)
print(processing_order)

{'C': {'A'}, 'D': {'A', 'B'}, 'E': {'C'}, 'F': {'C'}, 'G': {'D', 'E'}, 'H': {'F', 'G'}}
{'A': 2, 'B': 3, 'C': 2, 'D': 4, 'E': 4, 'F': 3, 'G': 5, 'H': 2}
{'E', 'H', 'A', 'F', 'C', 'B', 'D', 'G'}
('A', 'B', 'C', 'D', 'E', 'F', 'G', 'H')


In [4]:
#2. EARLIEST START AND EARLIEST FINISH WITH FORWARD PASS

earliest_start = {}
earliest_finish = {}

for activity in processing_order:
    if activity not in activities_with_predecessors:
        earliest_start[activity] = 0
        earliest_finish[activity] = time_duration[activity]
    else:
        predecessors = activities_with_predecessors[activity]
        maximum_predecessor_finish = max(earliest_finish[predecessor] for predecessor in predecessors)
        earliest_start[activity] = maximum_predecessor_finish
        earliest_finish[activity] = maximum_predecessor_finish + time_duration[activity]

print(earliest_start)
print(earliest_finish)

{'A': 0, 'B': 0, 'C': 2, 'D': 3, 'E': 4, 'F': 4, 'G': 8, 'H': 13}
{'A': 2, 'B': 3, 'C': 4, 'D': 7, 'E': 8, 'F': 7, 'G': 13, 'H': 15}


In [5]:
#3. LATEST START AND LATEST FINISH WITH BACKWARD PASS

inverse_processing_order = ('H', 'G', 'F', 'E', 'D', 'C', 'B', 'A')

latest_start = {}
latest_finish = {}

project_duration = max(earliest_finish.values())

for activity in inverse_processing_order:
    if activity == 'H':  
        latest_finish[activity] = project_duration
        latest_start[activity] = project_duration - time_duration[activity]
    else:
        successors = [act for act, preds in activities_with_predecessors.items() if activity in preds]
        minimum_successor_start = min(latest_start[successor] for successor in successors)
        latest_finish[activity] = minimum_successor_start
        latest_start[activity] = minimum_successor_start - time_duration[activity]

print(latest_start)
print(latest_finish)

{'H': 13, 'G': 8, 'F': 10, 'E': 4, 'D': 4, 'C': 2, 'B': 1, 'A': 0}
{'H': 15, 'G': 13, 'F': 13, 'E': 8, 'D': 8, 'C': 4, 'B': 4, 'A': 2}


In [6]:
#4. SLACK TIME AND CRITICAL PATH

for activity in processing_order:
    slack = latest_start[activity] - earliest_start[activity]
    critical = "YES" if slack == 0 else "NO"
    print(f"{activity:8}  {slack:3}     {critical}")

print(f"\nProject Duration: {project_duration} days")
critical_path = [a for a in processing_order if latest_start[a] - earliest_start[a] == 0]
print(f"Critical Path: {' → '.join(critical_path)}")

A           0     YES
B           1     NO
C           0     YES
D           1     NO
E           0     YES
F           6     NO
G           0     YES
H           0     YES

Project Duration: 15 days
Critical Path: A → C → E → G → H


In [7]:
#5. GANTT CHART

for activity in processing_order:
    start = earliest_start[activity]
    duration = time_duration[activity]
    slack = latest_start[activity] - earliest_start[activity]
    
    if slack == 0:
        marker = "█"  
    else:
        marker = "·"  
        
    line = " " * (start * 2) + marker * duration
    print(f"{activity}: {line}")

A: ██
B: ···
C:     ██
D:       ····
E:         ████
F:         ···
G:                 █████
H:                           ██
